In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.isnull().sum()
df.shape

(898, 35)

In [5]:
X = df.drop("Class",axis =1)
y = df["Class"]

In [6]:
df["Class"].unique() 

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)   # we trasfrom the multi class string output into int dtype

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train,y_test = train_test_split(
    X, y ,random_state = 42, test_size = 0.2
)

In [9]:
scaler = StandardScaler()
X_train_scaled =scaler.fit_transform(X_train)
X_test_scaled =scaler.transform(X_test)

# Apply PCA

In [25]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)  # keep 95% variance
X_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [26]:
print(X_train.shape)
print(X_test.shape)

(718, 34)
(180, 34)


# ANN

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


In [28]:
X_train_tensor = torch.tensor(X_pca,dtype = torch.float32)
y_train_tensor = torch.tensor(y_train,dtype = torch.long)

X_test_tensor = torch.tensor(X_test_pca,dtype = torch.float32)
y_test_tensor = torch.tensor(y_test,dtype = torch.long)


In [29]:
train_dataest = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [30]:
train_loader = DataLoader(train_dataest, batch_size = 32,shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32)


# Build our model

In [31]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_pca.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )
    def forward(self,x):
            return self.model(x)
    

In [32]:
model = ANN()

# optimizer & loss
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [33]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step()  # update param

        running_loss+=loss

    train_loss = running_loss/ len(train_loader)  # this is loss for one epoch

    print(f"epoch {epoch+1}/{epochs}, loss = {train_loss}")
        


epoch 1/100, loss = 1.6319507360458374
epoch 2/100, loss = 1.042677879333496
epoch 3/100, loss = 0.7202012538909912
epoch 4/100, loss = 0.5530222058296204
epoch 5/100, loss = 0.4524255692958832
epoch 6/100, loss = 0.3999509811401367
epoch 7/100, loss = 0.3525816798210144
epoch 8/100, loss = 0.33867502212524414
epoch 9/100, loss = 0.2976400554180145
epoch 10/100, loss = 0.28603142499923706
epoch 11/100, loss = 0.2622646391391754
epoch 12/100, loss = 0.24841254949569702
epoch 13/100, loss = 0.24382711946964264
epoch 14/100, loss = 0.23901410400867462
epoch 15/100, loss = 0.2238236367702484
epoch 16/100, loss = 0.22515524923801422
epoch 17/100, loss = 0.2076536864042282
epoch 18/100, loss = 0.20023922622203827
epoch 19/100, loss = 0.19618575274944305
epoch 20/100, loss = 0.1849866509437561
epoch 21/100, loss = 0.1980939507484436
epoch 22/100, loss = 0.17993170022964478
epoch 23/100, loss = 0.17760276794433594
epoch 24/100, loss = 0.17626117169857025
epoch 25/100, loss = 0.1750569343566894

In [34]:
# Evalution
model.eval()
total =0
correct =0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs  = model(xb) # [2.0, 0.5, 1.3, -0.5, ....]
        _, predicted = torch.max(outputs ,1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actula sample in one epoch

        
print("total :", total)
print("correct :", correct)
print("accuracy : ", correct/total * 100)

total : 180
correct : 168
accuracy :  93.33333333333333


In [35]:
y_test.shape

(180,)

In [36]:
y_train.shape

(718,)